# Temporal diagnostics and baseline model development

Reproduce the logistic regression baseline, measure changes across periods, and evaluate removing purchase month and year, adding promised lead time, and using a nonlinear model. Use Average Precision (AP) to select the model and F1 to select the threshold on validation data. Notebook 05 already evaluated this test set, so it is used here for retrospective diagnostics.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
split_dir = root / "data" / "processed" / "splits"
manifest = json.loads((split_dir / "split_manifest.json").read_text())
features = manifest["feature_columns"]
assert not set(features) & set(manifest["target_columns"] + manifest["metadata_columns"])

purchase = "order_purchase_timestamp"
delivery = "order_delivered_customer_date"
estimate = "order_estimated_delivery_date"
target = "is_delayed"
orders = pd.read_csv(
    root / "data" / "raw" / "olist_orders_dataset.csv",
    usecols=["order_id", purchase, delivery, estimate],
    parse_dates=[purchase, delivery, estimate],
)
orders["promised_days"] = (orders[estimate].dt.normalize() - orders[purchase].dt.normalize()).dt.days
frames = {}
for name, split in manifest["splits"].items():
    frame = pd.read_parquet(root / split["path"]).merge(
        orders.drop(columns=purchase), on="order_id", validate="one_to_one", how="left",
    )
    assert frame[[delivery, estimate]].notna().all().all()
    assert frame[target].eq(frame[delivery].dt.normalize().gt(frame[estimate].dt.normalize())).all()
    frames[name] = frame

train, validation, test = (frames[name] for name in ("train", "validation", "test"))
assert train[purchase].max() < validation[purchase].min()
assert validation[purchase].max() < test[purchase].min()
assert pd.concat(frames.values())["order_id"].is_unique

# The delivery outcome must already be observable when a period starts.
known_train = train.loc[train[delivery] < validation[purchase].min()]
known_validation = validation.loc[validation[delivery] < test[purchase].min()]
display(pd.DataFrame({
    "orders": [len(train), len(validation), len(test)],
    "delay_rate": [f[target].mean() for f in (train, validation, test)],
    "median_promised_days": [f["promised_days"].median() for f in (train, validation, test)],
    "labels_unavailable_at_next_cut": [len(train) - len(known_train), len(validation) - len(known_validation), np.nan],
    "observable_delay_rate": [known_train[target].mean(), known_validation[target].mean(), np.nan],
}, index=["train", "validation", "test"]).round(4))

,orders,delay_rate,median_promised_days,labels_unavailable_at_next_cut,observable_delay_rate
train,67529,0.0783,24.0,3100.0,0.0681
validation,14471,0.0431,25.0,1930.0,0.0428
test,14470,0.0428,18.0,NaN,NaN


## Changes over time

The monthly table splits months that span a partition boundary. The KS statistic measures the largest difference between the empirical cumulative distributions of observed values, including for discrete features. Changes in missingness are reported separately. Promised lead time is a candidate feature if the source field reflects the promise shown at checkout.

In [2]:
monthly = pd.concat([
    frame.assign(split=name, month=frame[purchase].dt.to_period("M"))
    for name, frame in frames.items()
]).groupby(["split", "month"], sort=False).agg(
    orders=(target, "size"), delay_rate=(target, "mean"),
    median_actual_days=("delivery_time_days", "median"),
    median_promised_days=("promised_days", "median"),
)
display(monthly.round(4))

drift = []
for feature in features + ["promised_days"]:
    reference = train[feature].astype(float)
    for name in ("validation", "test"):
        current = frames[name][feature].astype(float)
        drift.append({
            "feature": feature, "split": name,
            "ks": ks_2samp(reference.dropna(), current.dropna()).statistic,
            "missing_diff_pp": 100 * (current.isna().mean() - reference.isna().mean()),
        })
display(pd.DataFrame(drift).pivot(index="feature", columns="split").sort_values(
    ("ks", "test"), ascending=False,
).round(4))

coverage = orders.assign(
    month=orders[purchase].dt.to_period("M"),
    in_dataset=orders["order_id"].isin(pd.concat(frames.values())["order_id"]),
).groupby("month").agg(raw_orders=("order_id", "size"), included=("in_dataset", "mean"))
display(coverage.tail(6).round(4))

orders  delay_rate  median_actual_days  \
split      month                                             
train      2016-09       1      1.0000             54.8132   
           2016-10     265      0.0075             17.9443   
           2016-12       1      0.0000              4.6930   
           2017-01     750      0.0293             10.7336   
           2017-02    1653      0.0296             10.9712   
           2017-03    2546      0.0456             10.1265   
           2017-04    2303      0.0656             12.7486   
           2017-05    3545      0.0299              9.7933   
           2017-06    3135      0.0303             10.8049   
           2017-07    3872      0.0279             10.1581   
           2017-08    4193      0.0291              9.7698   
           2017-09    4150      0.0439             10.3144   
           2017-10    4478      0.0418             10.2634   
           2017-11    7288      0.1240             12.6987   
           2017-12    5513      0.0746             13.1422   
           2018-01    7069      0.0570             11.8301   
           2018-02    6555      0.1413             14.2638   
           2018-03    7003      0.1896             13.3677   
           2018-04    3209      0.0552              9.3866   
validation 2018-04    3589      0.0359              9.2497   
           2018-05    6749      0.0656              9.1479   
           2018-06    4133      0.0126              8.1236   
test       2018-06    1963      0.0097              7.2649   
           2018-07    6156      0.0338              7.5017   
           2018-08    6351      0.0619              7.0137   

                    median_promised_days  
split      month                          
train      2016-09                  19.0  
           2016-10                  55.0  
           2016-12                  27.0  
           2017-01                  39.0  
           2017-02                  31.0  
           2017-03                  23.0  
           2017-04                  26.0  
           2017-05                  24.0  
           2017-06                  24.0  
           2017-07                  24.0  
           2017-08                  23.0  
           2017-09                  23.0  
           2017-10                  23.0  
           2017-11                  23.0  
           2017-12                  28.0  
           2018-01                  26.0  
           2018-02                  25.0  
           2018-03                  22.0  
           2018-04                  23.0  
validation 2018-04                  25.0  
           2018-05                  22.0  
           2018-06                  29.0  
test       2018-06                  26.0  
           2018-07                  20.0  
           2018-08                  14.0

ks            missing_diff_pp           
split                       test validation            test validation
feature                                                               
purchase_year             0.6470     0.6470          0.0000      0.000
purchase_month            0.5129     0.4407          0.0000      0.000
promised_days             0.3368     0.0771          0.0000      0.000
total_freight             0.1814     0.1440          0.0000      0.000
all_sellers_same_state    0.0670     0.0515          0.0000      0.000
min_distance_km           0.0633     0.0500         -0.0908     -0.001
avg_distance_km           0.0625     0.0492         -0.0908     -0.001
max_distance_km           0.0623     0.0481         -0.0908     -0.001
purchase_weekday          0.0171     0.0513          0.0000      0.000
total_price               0.0124     0.0409          0.0000      0.000
purchase_hour             0.0097     0.0112          0.0000      0.000
items_quantity            0.0083     0.0070          0.0000      0.000
purchase_is_weekend       0.0069     0.0234          0.0000      0.000
sellers_quantity          0.0053     0.0062          0.0000      0.000
has_complete_coordinates  0.0009     0.0000          0.0000      0.000
unique_products           0.0007     0.0033          0.0000      0.000

,raw_orders,included
month,,
2018-05,6873,0.9820
2018-06,6167,0.9885
2018-07,6292,0.9784
2018-08,6512,0.9753
2018-09,16,0.0000
2018-10,4,0.0000


## Controlled experiments

The first row reproduces notebook 05. The second applies label availability cutoffs. The remaining runs use the same cutoffs to compare feature sets and algorithms. For direct comparison, the table also reports AP and F1 on the full validation set, with each protocol's threshold held fixed. The logistic regression runs evaluate removal of purchase month, purchase year, and both together.

In [3]:
def logistic():
    return make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(max_iter=2000, solver="lbfgs", random_state=42),
    )


def scores_for(model, columns, frame):
    return model.predict_proba(frame[columns].astype(float))[:, 1]


def choose_threshold(y, scores):
    precision, recall, thresholds = precision_recall_curve(y, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.argmax(f1)])


def evaluate(frame, scores, threshold):
    y = frame[target]
    pred = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "ap": average_precision_score(y, scores),
        "roc_auc": roc_auc_score(y, scores),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "alert_rate": pred.mean(),
        "fp": fp, "tp": tp,
    }


stable = [c for c in features if c not in {"purchase_month", "purchase_year"}]
with_promise = stable + ["promised_days"]
experiments = {
    "baseline_original": (features, logistic()),
    "baseline_with_available_labels": (features, logistic()),
    "logistic_without_year": ([c for c in features if c != "purchase_year"], logistic()),
    "logistic_without_month": ([c for c in features if c != "purchase_month"], logistic()),
    "logistic_without_year_and_month": (stable, logistic()),
    "logistic_with_promise": (with_promise, logistic()),
    "boosting_without_promise": (stable, HistGradientBoostingClassifier(random_state=42)),
    "boosting_with_promise": (with_promise, HistGradientBoostingClassifier(random_state=42)),
}
fitted = {}
rows = []
for name, (columns, model) in experiments.items():
    original = name == "baseline_original"
    fit_frame = train if original else known_train
    tune_frame = validation if original else known_validation
    model.fit(fit_frame[columns].astype(float), fit_frame[target])
    tune_scores = scores_for(model, columns, tune_frame)
    threshold = choose_threshold(tune_frame[target], tune_scores)
    fitted[name] = model, columns, threshold
    rows.append({
        "model": name, "threshold": threshold,
        "selection_ap": average_precision_score(tune_frame[target], tune_scores),
        **evaluate(validation, scores_for(model, columns, validation), threshold),
    })

results = pd.DataFrame(rows).set_index("model")
selected = results.drop(index="baseline_original")["selection_ap"].idxmax()
display(results.round(4))
print("Selected:", selected)
print("Selection prevalence:", round(known_validation[target].mean(), 4))

,threshold,selection_ap,ap,roc_auc,precision,recall,f1,alert_rate,fp,tp
model,,,,,,,,,,
baseline_original,0.2369,0.0597,0.0597,0.6060,0.0798,0.2244,0.1177,0.1212,1614,140
baseline_with_available_labels,0.1534,0.0613,0.0571,0.6010,0.0605,0.5994,0.1099,0.4273,5809,374
logistic_without_year,0.0893,0.0704,0.0679,0.6255,0.0908,0.2869,0.1380,0.1362,1792,179
logistic_without_month,0.1237,0.0707,0.0680,0.6261,0.0884,0.3061,0.1372,0.1493,1969,191
logistic_without_year_and_month,0.0886,0.0701,0.0676,0.6245,0.0892,0.2885,0.1363,0.1395,1838,180
logistic_with_promise,0.1250,0.1268,0.1283,0.7600,0.1634,0.3189,0.2161,0.0842,1019,199
boosting_without_promise,0.0900,0.0581,0.0588,0.5921,0.0710,0.3269,0.1166,0.1986,2670,204
boosting_with_promise,0.1245,0.1107,0.1134,0.7265,0.1327,0.3830,0.1971,0.1245,1562,239


Selected: logistic_with_promise
Selection prevalence: 0.0428


In [4]:
comparison = {}
# These prespecified contrasts diagnose the test drop without changing selection.
test_models = dict.fromkeys([
    "baseline_original", "baseline_with_available_labels",
    "logistic_without_month", "logistic_without_year_and_month", selected,
])
for name in test_models:
    model, columns, threshold = fitted[name]
    comparison[name] = evaluate(test, scores_for(model, columns, test), threshold)
display(pd.DataFrame(comparison).T.round(4))
print("Test prevalence:", round(test[target].mean(), 4))

# See whether the baseline's scores and alert volume shift month by month.
model, columns, threshold = fitted["baseline_original"]
baseline_monthly = []
for name, frame in frames.items():
    scores = scores_for(model, columns, frame)
    baseline_monthly.append(frame.assign(
        split=name, month=frame[purchase].dt.to_period("M"),
        score=scores, alert=scores >= threshold,
    ))
display(pd.concat(baseline_monthly).groupby(["split", "month"], sort=False).agg(
    orders=(target, "size"), delay_rate=(target, "mean"),
    mean_score=("score", "mean"), alert_rate=("alert", "mean"),
).tail(10).round(4))

,ap,roc_auc,precision,recall,f1,alert_rate,fp,tp
baseline_original,0.0408,0.4620,0.0317,0.3468,0.0582,0.4681,6559.0,215.0
baseline_with_available_labels,0.0412,0.4626,0.0285,0.3871,0.0531,0.5824,8187.0,240.0
logistic_without_month,0.0389,0.4241,0.0418,0.1516,0.0656,0.1554,2154.0,94.0
logistic_without_year_and_month,0.0389,0.4252,0.0434,0.1468,0.0669,0.1451,2008.0,91.0
logistic_with_promise,0.0680,0.6235,0.0488,0.2935,0.0837,0.2577,3547.0,182.0


Test prevalence: 0.0428


orders  delay_rate  mean_score  alert_rate
split      month                                              
train      2018-01    7069      0.0570      0.0998      0.0016
           2018-02    6555      0.1413      0.1162      0.0023
           2018-03    7003      0.1896      0.1293      0.0161
           2018-04    3209      0.0552      0.1436      0.0552
validation 2018-04    3589      0.0359      0.1454      0.0554
           2018-05    6749      0.0656      0.1616      0.0975
           2018-06    4133      0.0126      0.1867      0.2170
test       2018-06    1963      0.0097      0.1839      0.1982
           2018-07    6156      0.0338      0.2133      0.4868
           2018-08    6351      0.0619      0.2227      0.5335